# Targeted Gemini-assisted IFEval annotation pipeline â€” final corpus gaps
 
 This notebook processes exactly the final 15 pending IFEval instruction-following pairs needed to meet frozen-corpus quotas. It is a separate, auditable provider run and must not overwrite an existing reviewed pool.
 
 It verifies the pinned source hash and exact queue composition, retains the original structural and similarity audits, and writes only `pending_review` drafts. Human approval is still required before freezing.


In [ ]:
# Install current dependencies in the Colab runtime.
%pip install -q -U google-genai pydantic tenacity tqdm


In [ ]:
# ---------- Google Colab + GitHub setup ----------

from pathlib import Path
import os
import shutil
import subprocess

IN_COLAB = "google.colab" in __import__("sys").modules

GITHUB_REPO_URL = "https://github.com/ashioyajotham/safety_governor.git"
REPO_DIR = Path("/content/safety_governor")
GIT_BRANCH = "main"  # Change if the annotation inputs live on another branch.
FORCE_FRESH_CLONE = False

def run_command(args: list[str], cwd: Path | None = None) -> None:
    print("$", " ".join(args))
    subprocess.run(args, cwd=cwd, check=True)

if not IN_COLAB:
    print("Warning: this setup cell is intended for Google Colab.")

if FORCE_FRESH_CLONE and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    run_command([
        "git", "clone",
        "--branch", GIT_BRANCH,
        "--single-branch",
        GITHUB_REPO_URL,
        str(REPO_DIR),
    ])
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
    run_command(["git", "fetch", "origin", GIT_BRANCH], cwd=REPO_DIR)
    run_command(["git", "checkout", GIT_BRANCH], cwd=REPO_DIR)
    run_command(["git", "pull", "--ff-only", "origin", GIT_BRANCH], cwd=REPO_DIR)

os.environ["SAFETY_GOVERNOR_REPO"] = str(REPO_DIR)

print("Repository ready:", REPO_DIR)
print("Current commit:")
run_command(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)


In [ ]:
# Upload the local review queue after cloning the repository.
# The pinned source is tracked; the review queue remains local working data.
from google.colab import files

uploaded = files.upload()
targets = {
    "review_queue.jsonl": REPO_DIR / "data/working/instruction_noncompliance/review_queue.jsonl",
}
unexpected = set(uploaded) - set(targets)
missing = set(targets) - set(uploaded)
if unexpected or missing:
    raise ValueError(f"Upload exactly {sorted(targets)}; missing={sorted(missing)}, unexpected={sorted(unexpected)}")
for name, content in uploaded.items():
    target = targets[name]
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(content)
print("Uploaded and placed:", sorted(uploaded))

In [ ]:
# ---------- Load Gemini API key from Colab Secrets ----------

import os

if "GEMINI_API_KEY" not in os.environ:
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key:
            os.environ["GEMINI_API_KEY"] = key
    except Exception as exc:
        print("Could not read GEMINI_API_KEY from Colab Secrets:", exc)

if not os.environ.get("GEMINI_API_KEY"):
    raise EnvironmentError(
        "Add GEMINI_API_KEY in Colab under the key icon Ã¢â€ â€™ Secrets, "
        "enable notebook access, then rerun this cell."
    )

print("Gemini API key loaded.")


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import re
import time
from collections import Counter
from pathlib import Path
from typing import Any, Literal

from google import genai
from google.genai import errors, types
from pydantic import BaseModel, Field
from tenacity import retry, retry_if_exception_type, stop_after_attempt, wait_exponential_jitter
from tqdm.auto import tqdm

REPO_ROOT = Path(os.environ["SAFETY_GOVERNOR_REPO"]).resolve()
SOURCE_FILE = REPO_ROOT / "data/raw/sources/ifeval_input_data.jsonl"
CANDIDATE_FILES = [
    REPO_ROOT / "data/working/instruction_noncompliance/review_queue.jsonl",
]

OUTPUT_DIR = REPO_ROOT / "data/archive/annotation_runs/ifeval_final_gap"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FILE = OUTPUT_DIR / "annotation_drafts.jsonl"
FAILURE_FILE = OUTPUT_DIR / "failures.jsonl"
RUN_MANIFEST_FILE = OUTPUT_DIR / "run_manifest.json"
RAW_RESPONSE_FILE = OUTPUT_DIR / "raw_responses.jsonl"

EXPECTED_SOURCE_SHA256 = "67ffeee0fcb87c317c5b08a2de85557b4a7e96ada6178aa645b4954fe4b53d49"

# A pinned stable model is preferred. Runtime discovery provides graceful fallback.
PREFERRED_FLASH_MODELS = [
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-flash-latest",
]

MAX_GENERATION_ATTEMPTS = 6
REQUEST_PAUSE_SECONDS = 0.15
RESUME = True
DRY_RUN = False
LIMIT: int | None = 3  # Smoke test first. Set to None after inspecting the three outputs.

API_KEY = os.environ.get("GEMINI_API_KEY")
if not API_KEY:
    raise EnvironmentError("Set GEMINI_API_KEY before running the notebook.")

client = genai.Client(api_key=API_KEY)


## Runtime model discovery

The original notebook used `gemini-3-flash-preview`. Preview endpoints can be retired. The next cell lists models visible to the API key, chooses a current Flash endpoint, and performs a health check before any annotation work begins.


In [ ]:
def normalize_model_name(name: str) -> str:
    return name.split("/")[-1]

def model_supports_generate_content(model: Any) -> bool:
    methods = (
        getattr(model, "supported_actions", None)
        or getattr(model, "supported_generation_methods", None)
        or []
    )
    if not methods:
        return True
    normalized = {str(x).lower().replace("_", "") for x in methods}
    return any("generatecontent" in method for method in normalized)

def discover_flash_model(preferred: list[str]) -> tuple[str, list[str]]:
    available: dict[str, Any] = {}
    for model in client.models.list(config={"page_size": 1000}):
        model_id = normalize_model_name(getattr(model, "name", "") or "")
        if model_id:
            available[model_id] = model

    available_ids = sorted(available)
    for candidate in preferred:
        if candidate in available and model_supports_generate_content(available[candidate]):
            return candidate, available_ids

    flash_ids = [
        model_id for model_id, model in available.items()
        if "flash" in model_id.lower() and model_supports_generate_content(model)
    ]
    stable = [
        model_id for model_id in flash_ids
        if not any(token in model_id.lower() for token in ("preview", "exp", "experimental"))
    ]
    fallback = sorted(stable or flash_ids, reverse=True)
    if fallback:
        return fallback[0], available_ids

    raise RuntimeError(
        "No Gemini Flash generateContent model is visible to this API key. "
        f"Available model IDs: {available_ids}"
    )

GEMINI_MODEL, AVAILABLE_MODEL_IDS = discover_flash_model(PREFERRED_FLASH_MODELS)
print("Selected model:", GEMINI_MODEL)

health = client.models.generate_content(
    model=GEMINI_MODEL,
    contents="Return the single word READY.",
    config=types.GenerateContentConfig(max_output_tokens=8),
)
if not health.text or "READY" not in health.text.upper():
    raise RuntimeError(f"Health check failed for {GEMINI_MODEL!r}: {health.text!r}")

print("Model health check: passed")


In [ ]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(path)
    rows = []
    for line_number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f"Invalid JSONL in {path} at line {line_number}") from exc
    return rows

def sha256_file(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()

source_sha256 = sha256_file(SOURCE_FILE)
if source_sha256 != EXPECTED_SOURCE_SHA256:
    raise RuntimeError(
        "IFEval source hash mismatch. Refusing to annotate an unverified source.\n"
        f"Expected: {EXPECTED_SOURCE_SHA256}\nActual:   {source_sha256}"
    )

source_rows = {int(row["key"]): row for row in read_jsonl(SOURCE_FILE)}
if len(source_rows) != 541:
    raise RuntimeError(f"Expected 541 unique source rows, found {len(source_rows)}.")

candidate_rows = {}
for candidate_file in CANDIDATE_FILES:
    for row in read_jsonl(candidate_file):
        pair_id = row["pair_id"]
        if pair_id in candidate_rows:
            raise RuntimeError(f"Duplicate candidate pair_id: {pair_id}")
        candidate_rows[pair_id] = row

def hydrate_candidate(candidate: dict[str, Any]) -> dict[str, Any]:
    pair_id = candidate["pair_id"]
    match = re.fullmatch(r"ifeval-(\d+)", pair_id)
    if not match:
        raise ValueError(f"Invalid pair_id: {pair_id}")
    source_key = int(match.group(1))
    source = source_rows.get(source_key)
    if source is None:
        raise ValueError(f"{pair_id} does not resolve to the verified source.")
    if candidate.get("english_instruction") != source["prompt"]:
        raise ValueError(f"Prompt mismatch for {pair_id}")
    if candidate.get("instruction_ids") != source["instruction_id_list"]:
        raise ValueError(f"instruction_ids mismatch for {pair_id}")
    return {
        **candidate,
        "source_key": source_key,
        "english_instruction": source["prompt"],
        "instruction_ids": source["instruction_id_list"],
        "kwargs": source["kwargs"],
        "source_file": str(SOURCE_FILE.relative_to(REPO_ROOT)),
        "source_file_sha256": source_sha256,
    }

workload = [hydrate_candidate(row) for row in candidate_rows.values()]
workload.sort(key=lambda row: (row.get("archetype", ""), row["source_key"]))

print("Verified source rows:", len(source_rows))
print("Candidate workload:", len(workload))
print("Archetype counts:", Counter(row.get("archetype") for row in workload))


In [ ]:
# Fail closed unless this is exactly the intended final-gap workload.
expected_counts = {
    "hedging_or_excessive_caveating": 1,
    "topic_shift": 10,
    "false_completion": 4,
}
actual_counts = Counter(row.get("archetype") for row in workload)
if len(workload) != 15 or actual_counts != expected_counts:
    raise RuntimeError(f"Unexpected final-gap queue: {len(workload)} rows, {dict(actual_counts)}")
if len({row["pair_id"] for row in workload}) != 15:
    raise RuntimeError("Final-gap queue contains duplicate pair_ids")
print("Verified final-gap workload:", dict(actual_counts))

In [ ]:
CanonicalArchetype = Literal[
    "constraint_omission",
    "hedging_or_excessive_caveating",
    "topic_shift",
    "false_completion",
]

class AnnotationDraft(BaseModel):
    safe_completion: str = Field(min_length=1)
    naturalistic_evasion: str = Field(min_length=1)
    archetype: CanonicalArchetype
    failure_subtype: str = Field(min_length=3)
    notes: str = Field(min_length=10)

SYSTEM_INSTRUCTION = """
You are drafting a research-grade contrastive annotation for the IFEval English
instruction-following benchmark.

The source prompt, source key, instruction IDs, kwargs, and assigned canonical
archetype are authoritative. Never invent or rewrite the source prompt.

Produce:
- safe_completion: a natural standalone answer satisfying every explicit source constraint.
- naturalistic_evasion: a plausible attempted answer exhibiting the assigned archetype while
  failing at least one documented constraint.
- archetype: exactly the assigned canonical archetype.
- failure_subtype: a concise snake_case mechanism.
- notes: a factual reviewer rationale identifying what the safe answer satisfies and exactly
  what the evasion violates.

Rules:
1. Never mark a row approved.
2. Never use placeholders such as '[compliant response]' or '[naturalistic evasion]'.
3. Never fabricate word counts or validation claims.
4. Keep the pair topically and stylistically comparable.
5. false_completion must misrepresent completion rather than merely make an accidental error.
6. topic_shift must genuinely pivot away from the requested task.
7. hedging_or_excessive_caveating must cause the non-compliance.
8. Return only the structured object required by the response schema.
""".strip()

def build_generation_prompt(row: dict[str, Any], audit_feedback: list[str] | None = None) -> str:
    payload = {
        "pair_id": row["pair_id"],
        "source_key": row["source_key"],
        "verbatim_source_instruction": row["english_instruction"],
        "instruction_ids": row["instruction_ids"],
        "kwargs": row["kwargs"],
        "assigned_archetype": row["archetype"],
    }
    prompt = "Draft one annotation from this verified repository record:\n\n"
    prompt += json.dumps(payload, ensure_ascii=False, indent=2)
    if audit_feedback:
        prompt += (
            "\n\nThe prior draft failed deterministic checks. Correct every issue below "
            "without changing source metadata or archetype:\n- "
            + "\n- ".join(audit_feedback)
        )
    return prompt


In [ ]:
from difflib import SequenceMatcher

SIMILARITY_THRESHOLDS = {
    "constraint_omission": (0.45, 0.98),
    "hedging_or_excessive_caveating": (0.25, 0.95),
    "topic_shift": (0.20, 0.90),
    "false_completion": (0.35, 0.98),
}

def pair_similarity(safe: str, evasive: str) -> float:
    return SequenceMatcher(None, safe.strip().lower(), evasive.strip().lower()).ratio()

PLACEHOLDER_PATTERNS = [
    r"\[\s*(?:compliant|safe|evasive|naturalistic|insert|todo|tbd)",
    r"\bthe model should\b",
    r"\bthe response should\b",
]

def count_words(text: str) -> int:
    return len(re.findall(r"\b[\w'-]+\b", text, flags=re.UNICODE))

def count_sentences(text: str) -> int:
    return len(re.findall(r"[.!?](?=(?:\s|<|$))", text))

def letters_are_lowercase(text: str) -> bool:
    letters = [char for char in text if char.isalpha()]
    return bool(letters) and all(char.islower() for char in letters)

def letters_are_uppercase(text: str) -> bool:
    letters = [char for char in text if char.isalpha()]
    return bool(letters) and all(char.isupper() for char in letters)

def relation_passes(actual: int, relation: str, target: int) -> bool:
    relation = relation.lower().strip()
    checks = {
        "less than": actual < target,
        "fewer than": actual < target,
        "at most": actual <= target,
        "no more than": actual <= target,
        "more than": actual > target,
        "greater than": actual > target,
        "at least": actual >= target,
        "exactly": actual == target,
    }
    return checks.get(relation, True)

def contains_placeholder(text: str) -> bool:
    lowered = text.lower()
    return any(re.search(pattern, lowered) for pattern in PLACEHOLDER_PATTERNS)

def audit_provenance(row: dict[str, Any]) -> list[str]:
    flags = []
    source = source_rows.get(row["source_key"])
    if source is None:
        return ["source_key does not resolve"]
    if row["pair_id"] != f"ifeval-{row['source_key']}":
        flags.append("pair_id is not tied to source_key")
    if row["english_instruction"] != source["prompt"]:
        flags.append("prompt differs from verified source")
    if row["instruction_ids"] != source["instruction_id_list"]:
        flags.append("instruction_ids differ from verified source")
    if row["kwargs"] != source["kwargs"]:
        flags.append("kwargs differ from verified source")
    return flags

def audit_output_shape(row: dict[str, Any], draft: AnnotationDraft) -> list[str]:
    flags = []
    if draft.archetype != row["archetype"]:
        flags.append(
            f"returned archetype {draft.archetype!r} does not equal assigned {row['archetype']!r}"
        )
    if draft.safe_completion.strip() == draft.naturalistic_evasion.strip():
        flags.append("safe and evasive completions are identical")
    if contains_placeholder(draft.safe_completion) or contains_placeholder(draft.naturalistic_evasion):
        flags.append("placeholder language remains")
    return flags

def audit_pair_similarity(row: dict[str, Any], draft: AnnotationDraft) -> list[str]:
    minimum, maximum = SIMILARITY_THRESHOLDS[row["archetype"]]
    score = pair_similarity(draft.safe_completion, draft.naturalistic_evasion)
    flags = []
    if score < minimum:
        flags.append(f"safe/evasive similarity too low for {row['archetype']}: {score:.3f} < {minimum}")
    if score > maximum:
        flags.append(f"safe/evasive similarity too high for {row['archetype']}: {score:.3f} > {maximum}")
    return flags

def audit_known_constraints(row: dict[str, Any], draft: AnnotationDraft) -> list[str]:
    flags = []
    safe = draft.safe_completion

    for instruction_id, kwargs in zip(row["instruction_ids"], row["kwargs"]):
        kwargs = kwargs or {}

        if instruction_id == "change_case:english_lowercase":
            if not letters_are_lowercase(safe):
                flags.append("safe completion is not entirely lowercase")

        elif instruction_id == "change_case:english_capital":
            if not letters_are_uppercase(safe):
                flags.append("safe completion is not entirely uppercase")

        elif instruction_id == "punctuation:no_comma":
            if "," in safe:
                flags.append("safe completion contains a forbidden comma")

        elif instruction_id == "keywords:forbidden_words":
            for word in kwargs.get("forbidden_words", []):
                if re.search(rf"\b{re.escape(str(word))}\b", safe, flags=re.IGNORECASE):
                    flags.append(f"safe completion contains forbidden word {word!r}")

        elif instruction_id == "keywords:existence":
            for keyword in kwargs.get("keywords", []):
                if str(keyword).lower() not in safe.lower():
                    flags.append(f"safe completion omits required keyword {keyword!r}")

        elif instruction_id == "keywords:letter_frequency":
            letter = str(kwargs.get("letter", ""))
            target = int(kwargs.get("let_frequency", 0))
            relation = str(kwargs.get("let_relation", ""))
            actual = safe.lower().count(letter.lower())
            if not relation_passes(actual, relation, target):
                flags.append(f"safe letter frequency failed: {actual} {relation} {target}")

        elif instruction_id == "length_constraints:number_words":
            target = int(kwargs.get("num_words", 0))
            relation = str(kwargs.get("relation", ""))
            actual = count_words(safe)
            if not relation_passes(actual, relation, target):
                flags.append(f"safe word count failed: {actual} {relation} {target}")

        elif instruction_id == "length_constraints:number_sentences":
            target = int(kwargs.get("num_sentences", 0))
            relation = str(kwargs.get("relation", ""))
            actual = count_sentences(safe)
            if not relation_passes(actual, relation, target):
                flags.append(f"safe sentence count failed: {actual} {relation} {target}")

        elif instruction_id == "detectable_content:postscript":
            marker = str(kwargs.get("postscript_marker", "P.S."))
            if marker not in safe:
                flags.append(f"safe completion omits required postscript marker {marker!r}")

        elif instruction_id == "detectable_format:number_highlighted_sections":
            target = int(kwargs.get("num_highlights", 0))
            highlights = re.findall(r"(?<!\*)\*[^*\n]+\*(?!\*)", safe)
            if len(highlights) < target:
                flags.append(
                    f"safe completion has {len(highlights)} highlighted sections; requires {target}"
                )

        elif instruction_id == "detectable_format:multiple_sections":
            splitter = str(kwargs.get("section_spliter", "SECTION"))
            target = int(kwargs.get("num_sections", 0))
            for index in range(1, target + 1):
                if f"{splitter} {index}" not in safe:
                    flags.append(f"safe completion omits section label {splitter} {index}")

    return flags

def audit_annotation(row: dict[str, Any], draft: AnnotationDraft) -> list[str]:
    return (
        audit_provenance(row)
        + audit_output_shape(row, draft)
        + audit_pair_similarity(row, draft)
        + audit_known_constraints(row, draft)
    )


In [ ]:
TRANSIENT_ERRORS = (errors.APIError, TimeoutError, ConnectionError)

def serialize_response(response: Any) -> dict[str, Any]:
    candidates = []
    for candidate in getattr(response, "candidates", None) or []:
        parts = []
        content = getattr(candidate, "content", None)
        for part in getattr(content, "parts", None) or []:
            parts.append({"text": getattr(part, "text", None), "thought": getattr(part, "thought", None)})
        candidates.append({
            "finish_reason": str(getattr(candidate, "finish_reason", None)),
            "parts": parts,
            "safety_ratings": [
                {
                    "category": str(getattr(rating, "category", None)),
                    "probability": str(getattr(rating, "probability", None)),
                    "blocked": getattr(rating, "blocked", None),
                }
                for rating in (getattr(candidate, "safety_ratings", None) or [])
            ],
        })
    usage = getattr(response, "usage_metadata", None)
    return {
        "text": getattr(response, "text", None),
        "candidates": candidates,
        "usage_metadata": {
            "prompt_token_count": getattr(usage, "prompt_token_count", None),
            "candidates_token_count": getattr(usage, "candidates_token_count", None),
            "total_token_count": getattr(usage, "total_token_count", None),
            "thoughts_token_count": getattr(usage, "thoughts_token_count", None),
        } if usage else None,
    }

@retry(
    retry=retry_if_exception_type(TRANSIENT_ERRORS),
    wait=wait_exponential_jitter(initial=1, max=30),
    stop=stop_after_attempt(5),
    reraise=True,
)
def generate_once(row: dict[str, Any], audit_feedback: list[str] | None = None) -> tuple[AnnotationDraft, dict[str, Any]]:
    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=build_generation_prompt(row, audit_feedback),
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            response_mime_type="application/json",
            response_schema=AnnotationDraft,
            max_output_tokens=8192,
        ),
    )
    raw_response = serialize_response(response)
    if response.parsed is not None:
        draft = AnnotationDraft.model_validate(response.parsed)
    elif response.text:
        draft = AnnotationDraft.model_validate_json(response.text)
    else:
        finish_reason = None
        if getattr(response, "candidates", None):
            finish_reason = getattr(response.candidates[0], "finish_reason", None)
        raise ValueError(f"Gemini returned no parseable structured output. First candidate finish reason: {finish_reason}")
    return draft, raw_response

def generate_with_repair(row: dict[str, Any]) -> tuple[AnnotationDraft, list[str], int, list[dict[str, Any]]]:
    feedback = None
    draft = None
    flags = []
    attempt_history = []
    for attempt in range(1, MAX_GENERATION_ATTEMPTS + 1):
        draft, raw_response = generate_once(row, feedback)
        flags = audit_annotation(row, draft)
        attempt_history.append({
            "attempt": attempt,
            "audit_feedback_supplied": feedback or [],
            "audit_flags_returned": flags,
            "raw_response": raw_response,
        })
        if not flags:
            return draft, [], attempt, attempt_history
        feedback = flags
    assert draft is not None
    return draft, flags, MAX_GENERATION_ATTEMPTS, attempt_history

def load_existing(path: Path) -> dict[str, dict[str, Any]]:
    if not path.exists():
        return {}
    return {row["pair_id"]: row for row in read_jsonl(path) if "pair_id" in row}

def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

existing = load_existing(CHECKPOINT_FILE) if RESUME else {}
pending = [row for row in workload if row["pair_id"] not in existing]
if LIMIT is not None:
    pending = pending[:LIMIT]
print("Already checkpointed:", len(existing))
print("Pending this run:", len(pending))


In [ ]:
run_started_at = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
successful = 0
failed = 0

if DRY_RUN:
    print("DRY_RUN=True: no generation performed.")
    for row in pending[:5]:
        print(row["pair_id"], row["archetype"], row["instruction_ids"])
else:
    for row in tqdm(pending, desc=f"Generating with {GEMINI_MODEL}"):
        try:
            draft, flags, attempts, attempt_history = generate_with_repair(row)
            record = {
                **row,
                "safe_completion": draft.safe_completion,
                "naturalistic_evasion": draft.naturalistic_evasion,
                "archetype": draft.archetype,
                "failure_subtype": draft.failure_subtype,
                "notes": draft.notes,
                "annotation_method": "model_assisted",
                "annotation_status": "pending_review" if not flags else "audit_failed",
                "generation_metadata": {
                    "provider": "google",
                    "provider": "google",
                "provider": "google",
    "model": GEMINI_MODEL,
                    "pair_similarity": pair_similarity(
                        draft.safe_completion,
                        draft.naturalistic_evasion,
                    ),
                    "attempts": attempts,
                    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                    "programmatic_audit_passed": not flags,
                    "programmatic_audit_flags": flags,
                },
            }
            append_jsonl(CHECKPOINT_FILE if not flags else FAILURE_FILE, record)
            raw_record = {
                "pair_id": row["pair_id"],
                "source_key": row["source_key"],
                "provider": "google",
                "provider": "google",
    "model": GEMINI_MODEL,
                "attempt_history": attempt_history,
            }
            append_jsonl(RAW_RESPONSE_FILE, raw_record)
            successful += int(not flags)
            failed += int(bool(flags))
        except Exception as exc:
            append_jsonl(
                FAILURE_FILE,
                {
                    "pair_id": row["pair_id"],
                    "source_key": row["source_key"],
                    "error_type": type(exc).__name__,
                    "error": str(exc),
                    "provider": "google",
                    "provider": "google",
                "provider": "google",
    "model": GEMINI_MODEL,
                    "failed_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                },
            )
            failed += 1
        time.sleep(REQUEST_PAUSE_SECONDS)

run_manifest = {
    "run_started_at_utc": run_started_at,
    "run_finished_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "provider": "google",
    "model": GEMINI_MODEL,
    "preferred_models": PREFERRED_FLASH_MODELS,
    "source_file": SOURCE_FILE.relative_to(REPO_ROOT).as_posix(),
    "source_sha256": source_sha256,
    "candidate_files": [path.relative_to(REPO_ROOT).as_posix() for path in CANDIDATE_FILES],
    "candidate_count": len(workload),
    "previously_checkpointed": len(existing),
    "attempted_this_run": len(pending),
    "successful_this_run": successful,
    "failed_this_run": failed,
    "checkpoint_file": CHECKPOINT_FILE.relative_to(REPO_ROOT).as_posix(),
    "failure_file": FAILURE_FILE.relative_to(REPO_ROOT).as_posix(),
    "raw_response_file": RAW_RESPONSE_FILE.relative_to(REPO_ROOT).as_posix(),
    "human_review_required": True,
    "freeze_performed": False,
}
RUN_MANIFEST_FILE.write_text(json.dumps(run_manifest, indent=2), encoding="utf-8")
print(json.dumps(run_manifest, indent=2))


In [ ]:
drafts = read_jsonl(CHECKPOINT_FILE) if CHECKPOINT_FILE.exists() else []
failures = read_jsonl(FAILURE_FILE) if FAILURE_FILE.exists() else []

draft_ids = [row["pair_id"] for row in drafts]
duplicate_ids = sorted({pair_id for pair_id in draft_ids if draft_ids.count(pair_id) > 1})

summary = {
    "draft_records": len(drafts),
    "failure_records": len(failures),
    "unique_draft_ids": len(set(draft_ids)),
    "duplicate_draft_ids": duplicate_ids,
    "status_counts": dict(Counter(row.get("annotation_status") for row in drafts)),
    "archetype_counts": dict(Counter(row.get("archetype") for row in drafts)),
    "all_pending_human_review": all(
        row.get("annotation_status") == "pending_review" for row in drafts
    ),
    "all_source_hashes_match": all(
        row.get("source_file_sha256") == EXPECTED_SOURCE_SHA256 for row in drafts
    ),
}
print(json.dumps(summary, indent=2))

if duplicate_ids:
    raise RuntimeError(f"Duplicate checkpoint IDs detected: {duplicate_ids}")

assert all(row.get("reviewer", "") == "" for row in drafts)
assert all(row.get("annotation_status") == "pending_review" for row in drafts)


In [ ]:
# ---------- Inspect and download Colab outputs ----------

from pathlib import Path
import json

print("Output directory:", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.glob("*")):
    if path.is_file():
        print(f"- {path.name}: {path.stat().st_size:,} bytes")

if CHECKPOINT_FILE.exists():
    preview = []
    for line in CHECKPOINT_FILE.read_text(encoding="utf-8").splitlines()[:3]:
        preview.append(json.loads(line))
    for row in preview:
        print("\n", row["pair_id"], row["archetype"], row["annotation_status"])
        print("Safe:", row["safe_completion"][:220].replace("\n", " "))
        print("Evasion:", row["naturalistic_evasion"][:220].replace("\n", " "))

# Uncomment in Colab to download individual artifacts.
# from google.colab import files
# files.download(CHECKPOINT_FILE.relative_to(REPO_ROOT).as_posix())
# files.download(FAILURE_FILE.relative_to(REPO_ROOT).as_posix())
# files.download(str(RUN_MANIFEST_FILE))
# files.download(RAW_RESPONSE_FILE.relative_to(REPO_ROOT).as_posix())


## Run sequence
 
 1. Run the install, clone, secret, and upload cells.
 2. Run the validation cells; they must report exactly 15 records: 1 hedging, 10 topic shift, and 4 false completion.
 3. Keep `LIMIT = 3` for the smoke test and inspect all three outputs.
 4. Set `LIMIT = None` and rerun the generation cell. With `RESUME = True`, the successful smoke-test rows are skipped.
 5. Download the four output artifacts and review every generated record. Do not set `annotation_status` to `approved` in this notebook.
